# Dashboard Schedule Retry Health v1

本实验验证 `src/dashboard` 是否能把 `src/schedule` 的 retry metadata 变成 dashboard export 和 HTML 中可 review 的运行健康面板。

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

REPO_ROOT = next(
    path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (path / 'AGENTS.md').exists() and (path / 'src').exists()
)
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.dashboard import (
    build_dashboard_view_model,
    build_schedule_health,
    write_dashboard_html,
    write_model_matrix_dashboard_export,
)
from src.schedule import JobResult

CARD = REPO_ROOT / 'research/dashboard_schedule_retry_health_v1'
SOURCE_CARD = REPO_ROOT / 'research/factor_feature_set_matrix_v1'
SCHEDULE_HEALTH_PATH = CARD / 'schedule_health.json'
DASHBOARD_EXPORT = CARD / 'dashboard_export.json'
DASHBOARD_HTML = CARD / 'dashboard.html'
SUMMARY_PATH = CARD / 'analysis_summary.json'
CARD.mkdir(parents=True, exist_ok=True)

## 构造 schedule health

模拟一个永久失败的 dashboard refresh 和一个成功的数据更新 job，验证 retry history 会被标准化为 dashboard 可读结构。

In [2]:
failed_dashboard = JobResult(
    name='model_dashboard_refresh',
    status='failed',
    payload={
        'attempts': 2,
        'retry_schema_version': 'schedule.retry_attempts.v1',
        'retry_error_count': 2,
        'retry_errors': [
            {'attempt': 1, 'error_type': 'RuntimeError', 'message': 'transient dashboard failure'},
            {'attempt': 2, 'error_type': 'RuntimeError', 'message': 'permanent failure'},
        ],
    },
    detail='permanent failure',
)
successful_data_update = JobResult(
    name='data_update',
    status='success',
    payload={
        'attempts': 1,
        'retry_schema_version': 'schedule.retry_attempts.v1',
        'retry_error_count': 0,
        'retry_errors': [],
    },
)

schedule_health = build_schedule_health([failed_dashboard, successful_data_update])
SCHEDULE_HEALTH_PATH.write_text(json.dumps(schedule_health, ensure_ascii=False, indent=2), encoding='utf-8')

assert schedule_health['schema_version'] == 'dashboard.schedule_health.v1'
assert schedule_health['job_count'] == 2
assert schedule_health['failed_job_count'] == 1
assert schedule_health['total_attempts'] == 3
assert schedule_health['total_retry_error_count'] == 2
assert schedule_health['top_failure_job'] == 'model_dashboard_refresh'
assert schedule_health['top_failure_last_error'] == 'RuntimeError:permanent failure'
schedule_health

{'schema_version': 'dashboard.schedule_health.v1',
 'job_count': 2,
 'failed_job_count': 1,
 'total_attempts': 3,
 'total_retry_error_count': 2,
 'top_failure_job': 'model_dashboard_refresh',
 'top_failure_last_error': 'RuntimeError:permanent failure',
 'jobs': [{'name': 'model_dashboard_refresh',
   'status': 'failed',
   'attempts': 2,
   'retry_schema_version': 'schedule.retry_attempts.v1',
   'retry_error_count': 2,
   'last_error_attempt': 2,
   'last_error_type': 'RuntimeError',
   'last_error_message': 'permanent failure',
   'detail': 'permanent failure'},
  {'name': 'data_update',
   'status': 'success',
   'attempts': 1,
   'retry_schema_version': 'schedule.retry_attempts.v1',
   'retry_error_count': 0,
   'last_error_attempt': 0,
   'last_error_type': '',
   'last_error_message': '',
   'detail': ''}]}

## 接入 dashboard export 和 HTML

dashboard 只读取统一 export 中的 `schedule_health`，不在 view/page 阶段读取 sidecar 路径。

In [3]:
dashboard_payload = write_model_matrix_dashboard_export(
    SOURCE_CARD,
    output_path=DASHBOARD_EXPORT,
    top_n=3,
    schedule_health_path=SCHEDULE_HEALTH_PATH,
    allow_legacy_sidecars=True,
)
view_model = build_dashboard_view_model(dashboard_payload)
html = write_dashboard_html(view_model, output_path=DASHBOARD_HTML, title='SOXL Schedule Health Dashboard')

assert dashboard_payload['schedule_health'] == schedule_health
assert dashboard_payload['artifacts']['schedule_health'].endswith('schedule_health.json')
assert view_model['schedule_health']['top_failure_job'] == 'model_dashboard_refresh'
assert 'artifacts' not in view_model
assert 'Schedule Health' in html
assert 'attempts 2' in html
assert 'retry_errors 2' in html
assert 'RuntimeError:permanent failure' in html
assert 'schedule_health.json' not in html

dashboard_payload['schedule_health']

{'schema_version': 'dashboard.schedule_health.v1',
 'job_count': 2,
 'failed_job_count': 1,
 'total_attempts': 3,
 'total_retry_error_count': 2,
 'top_failure_job': 'model_dashboard_refresh',
 'top_failure_last_error': 'RuntimeError:permanent failure',
 'jobs': [{'name': 'model_dashboard_refresh',
   'status': 'failed',
   'attempts': 2,
   'retry_schema_version': 'schedule.retry_attempts.v1',
   'retry_error_count': 2,
   'last_error_attempt': 2,
   'last_error_type': 'RuntimeError',
   'last_error_message': 'permanent failure',
   'detail': 'permanent failure'},
  {'name': 'data_update',
   'status': 'success',
   'attempts': 1,
   'retry_schema_version': 'schedule.retry_attempts.v1',
   'retry_error_count': 0,
   'last_error_attempt': 0,
   'last_error_type': '',
   'last_error_message': '',
   'detail': ''}]}

## 汇总 sidecar

`analysis_summary.json` 记录本次 notebook 的核心验证点，HTML 和 JSON 作为 sidecar，中文结论仍保留在 notebook 最后一格。

In [4]:
summary = {
    'schema_version': 'research.dashboard_schedule_retry_health_v1.summary.v1',
    'experiment': 'dashboard_schedule_retry_health_v1',
    'schedule_health_schema_version': schedule_health['schema_version'],
    'job_count': schedule_health['job_count'],
    'failed_job_count': schedule_health['failed_job_count'],
    'total_attempts': schedule_health['total_attempts'],
    'total_retry_error_count': schedule_health['total_retry_error_count'],
    'top_failure_job': schedule_health['top_failure_job'],
    'top_failure_last_error': schedule_health['top_failure_last_error'],
    'dashboard_export': str(DASHBOARD_EXPORT),
    'dashboard_html': str(DASHBOARD_HTML),
    'promotion_decision': 'promote_to_src_dashboard_schedule_health',
    'risk_note': '该面板展示 schedule retry 元数据，不负责判断 provider/API 错误是否应该重试。',
}
SUMMARY_PATH.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary

{'schema_version': 'research.dashboard_schedule_retry_health_v1.summary.v1',
 'experiment': 'dashboard_schedule_retry_health_v1',
 'schedule_health_schema_version': 'dashboard.schedule_health.v1',
 'job_count': 2,
 'failed_job_count': 1,
 'total_attempts': 3,
 'total_retry_error_count': 2,
 'top_failure_job': 'model_dashboard_refresh',
 'top_failure_last_error': 'RuntimeError:permanent failure',
 'dashboard_export': '/Users/rziiiii/dev/workspace/research/dashboard_schedule_retry_health_v1/dashboard_export.json',
 'dashboard_html': '/Users/rziiiii/dev/workspace/research/dashboard_schedule_retry_health_v1/dashboard.html',
 'promotion_decision': 'promote_to_src_dashboard_schedule_health',
 'risk_note': '该面板展示 schedule retry 元数据，不负责判断 provider/API 错误是否应该重试。'}

## 中文结论

- 做了什么：新增 `src.dashboard.build_schedule_health`，并让 dashboard exporter、view model、HTML 页面消费 `schedule_health`，把 schedule retry metadata 展示为运行健康面板。
- 核心指标：本实验包含 2 个 job，1 个失败 job，`total_attempts=3`，`total_retry_error_count=2`，top failure 为 `model_dashboard_refresh`，最后错误为 `RuntimeError:permanent failure`。
- regime 差异：这里不是市场 regime，而是运行状态 regime；success job 没有 retry error，permanent failure job 保留 attempts、retry_errors 和 last_error，dashboard 可直接区分 transient/retry 后失败路径。
- 是否晋升：晋升到 `src/dashboard`，因为 schedule、dashboard export 和 notebook review 都需要统一读取这一结构；`src/schedule` 只负责产出 JobResult/retry payload，不承担页面展示。
- 风险：该面板不替代 provider/API 错误分类，也不决定是否继续重试；后续需要把真实定时运行产物定期写入 `schedule_health.json`。
- 下一步：把 scheduler 多 job 运行结果聚合写入固定 schedule health artifact，并在 production health/readiness overview 中增加 schedule gate。